# Roman Detector Cutouts With Gladius on Nexus

## Learning Goals

By the end of this tutorial, you will be able to:

- Select an accessible calibrated Roman detector image through MAST.
- Create native cutouts from sky positions or detector-pixel coordinates.
- Inspect science values, uncertainty arrays and data-quality flags.
- Locate original metadata and verify the cutout's pixel-to-sky coordinates.
- Build or modify the same configuration using the optional embedded GUI.

## Introduction

A cutout is a smaller image copied from an input detector image. Gladius keeps
the original sampled pixel values and separately records the original image's
metadata. This tutorial queries the Mikulski Archive for Space Telescopes (MAST)
for a real calibrated Roman image; it does not substitute synthetic data.
The Roman Space Telescope's Wide Field Instrument (WFI) records detector images;
"calibrated" means these are pipeline-processed images rather than raw detector
reads. The example coordinate is the chosen image's archive centre, not an
identified or measured star.

**Audience:** astronomers comfortable with basic Python and Jupyter notebooks.
The tutorial is designed for Roman Research Nexus. The Python workflow can also
run elsewhere; its optional embedded GUI section requires the Nexus proxy.
It is independent community software, not an STScI-endorsed tool.

**Data and resources:** you need an authorised MAST account, an explicitly chosen
credential file, and space for one full detector image. The default advertised
size limit is 512 MiB. The actual download and runtime depend on the selected
image, connection and server. Check the metadata and size before the download
cell. Reduced cutouts can retain restricted pointing and other metadata; do not
publish their outputs while the original observations are restricted.

Install `roman-gladius[roman,mast,notebook]` in the selected Python kernel.
The sibling `requirements.txt` records the dependency baseline. The main Python
workflow and optional GUI require Gladius 0.1.0a3 or later. The GUI stays
disabled by default; enable it when running interactively.
Run cells in order after setting the configuration below. The notebook is kept
separate from RTD and committed with no saved outputs.


## 1 · Configure your session

**Required:** set `ENV_FILE` to an existing file containing `MAST_API_TOKEN=...`.
Do not paste the token into a notebook cell. This explicitly selected file supplies
the token for the session; there is no search of other directories.

Choose a writable scratch workspace with enough space for a full detector image.
The parent is staged temporarily during extraction; the cutouts and run records
remain in `WORKSPACE/gladius-mast-test`.

`TARGET_CSV=None` uses the selected image's archive centre. That is a convenient
sky coordinate, **not a source identification**. Supply your own CSV if you have
targets known to fall on the selected image. CSV columns are `id,ra,dec` in ICRS
degrees, or `id,x,y` in zero-based parent pixels; extra columns are retained.
Create the token at [MAST API tokens](https://auth.mast.stsci.edu/tokens).
A token identifies your account; it does not itself grant restricted-data access.
For automated execution, set `GLADIUS_TUTORIAL_ENV_FILE` to the path of an
existing credential file before starting the kernel. This selects a file
explicitly; the notebook does not search directories or embed the token.

Leave `RUN_GUI=False` for automated execution and static exports. After completing
the Python example, set it to `True` only if you want to use the live GUI.


In [ ]:
import os
from pathlib import Path

ENV_FILE = Path(os.environ["GLADIUS_TUTORIAL_ENV_FILE"]) if os.environ.get("GLADIUS_TUTORIAL_ENV_FILE") else None
# Or replace None above with Path("/your/private/location/.env"). Never paste a token here.
WORKSPACE = Path.cwd() / "gladius-example"
RUN_GUI = False  # Optional interactive section; keep False for unattended runs.
TARGET_CSV = None  # Optional Path to your own CSV of targets on the chosen image.
CUTOUT_SIZE = 150  # A 150 × 150 array of native detector pixels.
MAX_PARENT_MIB = 512  # Reject larger advertised files before download.
WCS_ROUNDTRIP_TOLERANCE = 0.01  # Pixels; explicit notebook choice, package default is 0.001.

In [ ]:
import os
from dotenv import dotenv_values

if ENV_FILE is None:
    raise ValueError("Set ENV_FILE to the path of your local credential file, then rerun this cell.")
ENV_FILE = Path(ENV_FILE).expanduser().resolve()
if not ENV_FILE.is_file():
    raise FileNotFoundError("ENV_FILE must point to an existing file; no archive request was made.")
credentials = dotenv_values(ENV_FILE, interpolate=False)
if not (credentials.get("MAST_API_TOKEN") or "").strip():
    raise ValueError("The selected file must contain a nonempty MAST_API_TOKEN.")
os.environ["MAST_API_TOKEN"] = credentials["MAST_API_TOKEN"]
del credentials

WORKSPACE = Path(WORKSPACE).expanduser().resolve()
if not isinstance(CUTOUT_SIZE, int) or isinstance(CUTOUT_SIZE, bool) or CUTOUT_SIZE < 1:
    raise ValueError("CUTOUT_SIZE must be a positive integer.")
if not isinstance(MAX_PARENT_MIB, (int, float)) or MAX_PARENT_MIB <= 0:
    raise ValueError("MAX_PARENT_MIB must be positive.")
print("Credential file loaded. Token is not displayed.")

## Libraries Used

Python's `pathlib`, `os` and `json` handle paths, the explicitly selected credential
file and run records. `python-dotenv` reads that file without evaluating shell
commands. Gladius provides archive selection, configuration, extraction and
readers. NumPy checks image arrays and flags; Astropy provides tables, coordinate
conversions and display normalization. Matplotlib displays science and uncertainty
arrays. ASDF opens the saved metadata, and PyYAML writes a reusable configuration.
The optional GUI uses Python's `threading` and IPython's `IFrame` display.

In the output, **SCI** means the science pixel values, **ERR** the per-pixel
uncertainties and **DQ** the integer data-quality flags. **WCS** (World Coordinate
System) converts between pixels and positions on the sky; **GWCS** is the general
model representation Gladius retains. **ICRS** is the celestial reference frame
used for the supplied RA/Dec angles, measured in degrees. **ASDF** and **FITS** are
the two output file formats. See the [Gladius scientific conventions](https://roman-gladius.readthedocs.io/en/latest/science.html)
for worked pixel and coordinate examples.


## 2 · Select one real calibrated image

This uses the same metadata-only selection as the GUI's **Load test**: sample up
to five Level 2 datasets and select the smallest known-size calibrated product
among them. It does not claim to find the archive's smallest image. No image is
downloaded in this cell.

Inspect the filename, advertised size and target positions before running the
extraction cell. The size check uses archive metadata, not a streaming byte quota.
An inaccessible archive or an empty calibrated sample raises an error instead of
substituting fabricated data.

In [ ]:
import json
import numpy as np
from astropy.table import Table
from IPython.display import display

from gladius import RunConfig, read_targets_csv, run
from gladius.inputs import mast_test_configuration
from gladius.io import read_cutout


def show_table(headers, rows):
    display(Table(rows=rows, names=headers))


selection = mast_test_configuration(WORKSPACE)
advertised_mib = selection["download_bytes"] / 1024**2
show_table(["Dataset", "Calibrated parent", "Advertised MiB"], [
    [selection["dataset"], selection["filename"], f"{advertised_mib:.1f}"]
])
if advertised_mib > MAX_PARENT_MIB:
    raise ValueError("Selected parent exceeds MAX_PARENT_MIB. No image was downloaded; choose a deliberate larger limit or a different sample.")

settings = selection["configuration"]
settings["extraction"].update(size=CUTOUT_SIZE, no_overlap="skip")
settings["astrometry"]["roundtrip_tolerance_pixels"] = WCS_ROUNDTRIP_TOLERANCE
settings["storage"]["formats"] = ["asdf", "fits"]
settings["stacks"]["enabled"] = False  # One image does not need a display stack.
if TARGET_CSV is not None:
    settings["targets"] = [target.model_dump(mode="json") for target in read_targets_csv(TARGET_CSV)]
config = RunConfig.model_validate(settings).resolved(WORKSPACE)
show_table(["Target", "Coordinates", "Metadata fields"], [
    [t.id, f"{t.ra:.7f}, {t.dec:.7f} deg" if t.ra is not None else f"{t.x}, {t.y} parent pixels", ", ".join(t.metadata) or "—"]
    for t in config.targets
])

## 3 · Make the cutouts

**This cell downloads the selected full parent image.** Gladius opens that image
once, processes all targets, and removes only its own staged download afterward.
Existing local inputs are never deleted. Outputs retain SCI/ERR/DQ and parent
metadata. The numerical WCS check can reject an inconsistent transform; it cannot
establish absolute sky accuracy. This notebook explicitly uses a 0.01-pixel
round-trip tolerance (the package default is 0.001). A real sampled parent had
about 0.0034 pixels of numerical inverse error; this is why the notebook makes
the tolerance visible in the configuration cell. This does not correct or
certify the supplied sky solution.

The output manifest determines which products completed. Non-overlapping targets
are skipped in this example. If none overlap, choose a matching target/image
sample rather than treating an empty result as success.

In [ ]:
import yaml

WORKSPACE.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "example-config.yaml").write_text(
    yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False), encoding="utf-8"
)
manifest_path = run(config, progress=print)
run_directory = manifest_path.parent
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
entries = list(manifest["entries"].values())
completed = [entry for entry in entries if entry["status"] == "complete"]
show_table(["Run status", "Completed", "Skipped", "Failed"], [[
    manifest["status"], len(completed),
    sum(e["status"] == "skipped" for e in entries),
    sum(e["status"] == "failed" for e in entries),
]])
if manifest["status"] != "complete":
    raise RuntimeError("Run failed. Inspect manifest['errors'] locally before continuing.")
if not completed:
    raise RuntimeError("No targets overlapped this image. Supply a matching target CSV/image sample.")
print("Manifest:", manifest_path.relative_to(WORKSPACE))

## 4 · Inspect the native image planes

Each row below is one cutout: **SCI**, **ERR**, and the fraction of pixels carrying
each of the first two DQ bits. NaN padding remains NaN; padded DQ pixels carry
DO_NOT_USE. The plotted science stretch is for display only and does not alter
the arrays. At most four targets are plotted to keep notebook output manageable.

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import AsinhStretch, ImageNormalize, PercentileInterval

cutouts = []
for entry in completed[:4]:
    filename = next(name for name in entry["files"] if name.endswith(".asdf"))
    cutouts.append((entry, read_cutout(run_directory / filename)))

with plt.rc_context({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False}):
    fig, axes = plt.subplots(len(cutouts), 2, figsize=(10, 4.2 * len(cutouts)), squeeze=False, constrained_layout=True)
    fig.set_facecolor("#fbf9fd")
    for row, (entry, cutout) in enumerate(cutouts):
        for col, (plane, label) in enumerate((("data", "SCI"), ("err", "ERR"))):
            values = cutout.arrays[plane]
            finite = values[np.isfinite(values)]
            if not finite.size:
                raise ValueError("Cutout has no finite values to display; inspect overlap and DQ.")
            norm = ImageNormalize(finite, interval=PercentileInterval(99), stretch=AsinhStretch())
            ax = axes[row, col]
            im = ax.imshow(values, origin="lower", cmap="magma", norm=norm, interpolation="nearest")
            ax.set_title(f"{entry['target_id']} · {label}", loc="left", color="#62398b", fontweight="bold")
            ax.set_xlabel("Cutout x (pixel)")
            ax.set_ylabel("Cutout y (pixel)")
            fig.colorbar(im, ax=ax, fraction=.046, pad=.04, label=entry.get("data_units") or "Native units")
    plt.show()

show_table(["Target", "Shape", "DO_NOT_USE", "SATURATED"], [
    [entry["target_id"], str(cutout.arrays["data"].shape),
     f"{np.mean((cutout.arrays['dq'] & 1) != 0):.1%}",
     f"{np.mean((cutout.arrays['dq'] & 2) != 0):.1%}"]
    for entry, cutout in cutouts
])

## 5 · Check the metadata and exact sky transform

The original parent footprint is provenance, not the cutout's own footprint.
Gladius keeps the parent tree under `parent_metadata`; derived coverage is stored
separately in `cutout_metadata`. The exact translated WCS is available through
`read_cutout` for both ASDF and Gladius FITS products.

Here we compare ASDF/FITS science arrays and inspect only selected metadata.
This is an output consistency check, not external astrometric calibration.

In [ ]:
import asdf
from gladius.core import pixel_to_world

entry, cutout = cutouts[0]
asdf_name = next(name for name in entry["files"] if name.endswith(".asdf"))
fits_name = next(name for name in entry["files"] if name.endswith(".fits"))
fits_cutout = read_cutout(run_directory / fits_name)
for plane in ("data", "err", "dq"):
    np.testing.assert_array_equal(cutout.arrays[plane], fits_cutout.arrays[plane])

with asdf.open(run_directory / asdf_name, lazy_load=False, memmap=False) as af:
    derived = dict(af.tree["cutout_metadata"])

ny, nx = cutout.arrays["data"].shape
ra, dec = pixel_to_world(cutout.wcs, (nx - 1) / 2, (ny - 1) / 2)
show_table(["Property", "Value"], [
    ["ASDF / FITS native planes", "Exactly equal"],
    ["Original parent metadata sections", ", ".join(cutout.parent_metadata)],
    ["Pixel mapping", json.dumps(cutout.mapping, default=str)],
    ["Centre from translated WCS (ICRS deg)", f"{float(ra):.8f}, {float(dec):.8f}"],
    ["Cutout valid footprint", derived.get("s_region")],
    ["Virtual grid footprint", derived.get("virtual_s_region")],
])

## 6 · Repeat the run

Resume is enabled by default. With matching configuration, code/dependency
versions, input identities and output checksums, completed cutouts are reused.
Missing or corrupt products are made again. MAST discovery and parent staging can
still occur, so this cell may **download the same parent again**; resume is not a
network cache. Run it only if you want to demonstrate that behaviour.

In [ ]:
REPEAT_RUN = False

if REPEAT_RUN:
    repeated_path = run(config, progress=print)
    repeated = json.loads(repeated_path.read_text(encoding="utf-8"))
    assert repeated_path == manifest_path
    assert repeated["status"] == "complete"
    print("Same run identity; see progress messages for reused cutouts.")
else:
    print("Repeat skipped. Set REPEAT_RUN=True to try resume.")

## 7 · Use the GUI inside this notebook

The GUI can explain settings, build YAML, and run the same pipeline. This example
starts a server in this kernel and embeds it below using the authenticated Nexus
proxy. It requires Gladius 0.1.0a3 or later.

Set `RUN_GUI=True` in the configuration cell, then run the next cell when you
want to open the GUI. With the default `False`, the section is skipped cleanly
for automated execution and static exports. It opens with the configuration from the Python example already loaded,
without starting a new cutout run. Review the settings, or use **Load test** or
add your own targets. Click **Make cutouts** only when ready. It reads and writes files on Nexus, just like the Python workflow.

The frame uses the same Nexus origin as the notebook. If it appears blank, open
the printed link in a separate tab and check that you are signed into the same
Nexus server. If the proxy extension is unavailable, ask the Nexus administrators;
there is no need to expose a public Gladius port.


In [ ]:
if RUN_GUI:
    import threading

    from IPython.display import IFrame, display
    from gladius.web import GuiServer

    if "gui_thread" not in globals() or not gui_thread.is_alive():
        # Port 0 selects a free port; the proxy URL retains your named server.
        gui_server = GuiServer(
            WORKSPACE, port=0, nexus=True, iframe_origin="self", initial_config=config
        )
        gui_thread = threading.Thread(target=gui_server.serve_forever, daemon=True)
        gui_thread.start()

    gladius_ui_url = gui_server.public_url
    print("Open the GUI in a separate tab:", gladius_ui_url)
    display(IFrame(src=gladius_ui_url, width="100%", height=650))
else:
    print("Interactive GUI skipped. Set RUN_GUI=True to explore it on Nexus.")


### Stop the embedded GUI

Run this cell when finished. Closing the frame does not stop the server or a
running pipeline. Wait for **Complete** or **Failed** before stopping it. Restarting
the notebook kernel also stops this embedded server.


In [ ]:
if "gui_thread" in globals() and gui_thread.is_alive():
    gui_server.shutdown()
    gui_server.server_close()
    gui_thread.join(timeout=5)
    print("Embedded GUI stopped.")


## Keep or share the right things

- Native products, the resolved YAML and manifest are in your selected workspace.
- ASDF/FITS preserve parent and target metadata, including pointing information.
  Keep results private while their inputs are restricted.
- The YAML contains configuration, not your MAST token. Keep the `.env` private.
- Before sharing or committing this notebook, **clear every cell output**.
  Do not use plots or metadata from this run as public examples until the data
  themselves are cleared for release.

For batch use, reuse the saved YAML with `gladius run example-config.yaml` in an
environment with your token available. In the CLI, `.env` is loaded beside that
YAML; the explicit `ENV_FILE` path above is a notebook setting.

## Summary

You selected a real calibrated image, made native cutouts in ASDF and FITS,
inspected the science/uncertainty arrays and quality flags, and checked that the
two saved formats recover the same pixel values. The mapping records how each
cutout pixel corresponds to the original detector image. The original metadata
and the cutout's new footprint have separate saved locations.

The workflow does not identify a star, fit a centroid, correct astrometry against
Gaia, or establish that the image is suitable for a particular scientific
measurement. Retaining metadata preserves information; it does not prove that
the original calibration or coordinate model is accurate.

## Additional Resources

- [Gladius configuration and execution](https://roman-gladius.readthedocs.io/en/latest/configuration.html)
- [Gladius headers and metadata](https://roman-gladius.readthedocs.io/en/latest/headers.html)
- [Roman Research Nexus](https://roman-docs.stsci.edu/data-handbook/roman-research-nexus)
- [MAST Roman documentation](https://archive.stsci.edu/missions-and-data/roman)

## About This Notebook

Authors: Gladius contributors. Last revised: 2026-11-01. License: BSD-3-Clause.
Questions and corrections: [Gladius issues](https://github.com/AmberLee2427/gladius/issues).
No STScI affiliation, acceptance or endorsement is claimed.

The released 0.1.0a2 package passed synthetic functional checks on Nexus with
Python 3.13 and the Framework with Python 3.12. Those checks are not a full
execution of this real-MAST notebook. Automated execution additionally needs an
authorised credential file and data that the chosen account can access. The GUI
extension remains disabled until explicitly requested and needs a newer Gladius
version than the Python-only example.

## Citations

For Gladius, cite the package name, version and repository; see
[its citation metadata](https://github.com/AmberLee2427/gladius/blob/main/CITATION.cff).
An archival DOI has not been assigned. Follow the project guidance for
[citing Astropy](https://www.astropy.org/acknowledging.html),
[NumPy](https://numpy.org/citing-numpy/),
[SciPy](https://scipy.org/citing-scipy/) and
[Matplotlib](https://matplotlib.org/stable/project/citing.html) when used in research.
Follow the applicable archive and mission guidance when publishing results from
Roman/MAST data. No data citation or observation identity is invented here.
